In [1]:
import pandas as pd
import numpy as np
import os
import glob
import joblib

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:


tourist = pd.read_csv("Top Indian Places to Visit.csv")

print("Shape:", tourist.shape)
tourist.head()

Shape: (325, 16)


,Unnamed: 0,Zone,State,City,Name,Type,Establishment Year,time needed to visit in hrs,Google review rating,Entrance Fee in INR,Airport with 50km Radius,Weekly Off,Significance,DSLR Allowed,Number of google review in lakhs,Best Time to visit
0,0,Northern,Delhi,Delhi,India Gate,War Memorial,1921,0.5,4.6,0,Yes,NaN,Historical,Yes,2.60,Evening
1,1,Northern,Delhi,Delhi,Humayun's Tomb,Tomb,1572,2.0,4.5,30,Yes,NaN,Historical,Yes,0.40,Afternoon
2,2,Northern,Delhi,Delhi,Akshardham Temple,Temple,2005,5.0,4.6,60,Yes,NaN,Religious,No,0.40,Afternoon
3,3,Northern,Delhi,Delhi,Waste to Wonder Park,Theme Park,2019,2.0,4.1,50,Yes,Monday,Environmental,Yes,0.27,Evening
4,4,Northern,Delhi,Delhi,Jantar Mantar,Observatory,1724,2.0,4.2,15,Yes,NaN,Scientific,Yes,0.31,Morning


In [3]:
print(tourist.columns.tolist())

['Unnamed: 0', 'Zone', 'State', 'City', 'Name', 'Type', 'Establishment Year', 'time needed to visit in hrs', 'Google review rating', 'Entrance Fee in INR', 'Airport with 50km Radius', 'Weekly Off', 'Significance', 'DSLR Allowed', 'Number of google review in lakhs', 'Best Time to visit']


In [4]:
tourist.columns = (
    tourist.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)

print(tourist.columns.tolist())

['unnamed:_0', 'zone', 'state', 'city', 'name', 'type', 'establishment_year', 'time_needed_to_visit_in_hrs', 'google_review_rating', 'entrance_fee_in_inr', 'airport_with_50km_radius', 'weekly_off', 'significance', 'dslr_allowed', 'number_of_google_review_in_lakhs', 'best_time_to_visit']


In [5]:
tourist.isnull().sum().sort_values(ascending=False)

weekly_off                          293
unnamed:_0                            0
state                                 0
zone                                  0
name                                  0
type                                  0
establishment_year                    0
city                                  0
time_needed_to_visit_in_hrs           0
google_review_rating                  0
entrance_fee_in_inr                   0
airport_with_50km_radius              0
significance                          0
dslr_allowed                          0
number_of_google_review_in_lakhs      0
best_time_to_visit                    0
dtype: int64

In [6]:
for col in tourist.columns:
    print(col, ":", tourist[col].dtype)

unnamed:_0 : int64
zone : str
state : str
city : str
name : str
type : str
establishment_year : str
time_needed_to_visit_in_hrs : float64
google_review_rating : float64
entrance_fee_in_inr : int64
airport_with_50km_radius : str
weekly_off : str
significance : str
dslr_allowed : str
number_of_google_review_in_lakhs : float64
best_time_to_visit : str


In [7]:
text_columns = []

for col in ["city", "name", "type", "tags", "description", "category"]:
    if col in tourist.columns:
        text_columns.append(col)

print("Using:", text_columns)

Using: ['city', 'name', 'type']


In [8]:
tourist["combined_text"] = tourist[text_columns].fillna("").astype(str).agg(" ".join, axis=1)

tourist[["combined_text"]].head()

,combined_text
0,Delhi India Gate War Memorial
1,Delhi Humayun's Tomb Tomb
2,Delhi Akshardham Temple Temple
3,Delhi Waste to Wonder Park Theme Park
4,Delhi Jantar Mantar Observatory


In [9]:
tfidf = TfidfVectorizer(
    stop_words="english",
    ngram_range=(1, 2)
)

tfidf_matrix = tfidf.fit_transform(tourist["combined_text"])

print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (325, 1561)


In [10]:
similarity_matrix = cosine_similarity(tfidf_matrix)

print(similarity_matrix.shape)

(325, 325)


In [11]:
def recommend_places(city=None, trip_type=None, top_n=10):
    
    data = tourist.copy()
    
    if city:
        city_mask = data["city"].astype(str).str.contains(
            city,
            case=False,
            na=False
        )
        data = data[city_mask]
    
    if data.empty:
        return pd.DataFrame()
    
    query_parts = []
    
    if city:
        query_parts.append(city)
    
    if trip_type:
        query_parts.append(trip_type)
    
    query = " ".join(query_parts)
    
    if query:
        query_vector = tfidf.transform([query])
        scores = cosine_similarity(query_vector, tfidf_matrix).flatten()
    else:
        scores = np.zeros(len(tourist))
    
    tourist_copy = tourist.copy()
    tourist_copy["score"] = scores
    
    if city:
        tourist_copy = tourist_copy[
            tourist_copy["city"].astype(str).str.contains(
                city,
                case=False,
                na=False
            )
        ]
    
    return tourist_copy.sort_values(
        "score",
        ascending=False
    ).head(top_n)

In [12]:
recommend_places(
    city="Ranchi",
    trip_type="adventure",
    top_n=10
)

,unnamed:_0,zone,state,city,name,type,establishment_year,time_needed_to_visit_in_hrs,google_review_rating,entrance_fee_in_inr,airport_with_50km_radius,weekly_off,significance,dslr_allowed,number_of_google_review_in_lakhs,best_time_to_visit,combined_text,score
297,297,Eastern,Jharkhand,Ranchi,Pahari Mandir,Temple,Unknown,1.0,4.6,0,Yes,NaN,Religious,Yes,0.13,All,Ranchi Pahari Mandir Temple,0.299164


In [13]:
recommend_places("Delhi", "historical", 10)

,unnamed:_0,zone,state,city,name,type,establishment_year,time_needed_to_visit_in_hrs,google_review_rating,entrance_fee_in_inr,airport_with_50km_radius,weekly_off,significance,dslr_allowed,number_of_google_review_in_lakhs,best_time_to_visit,combined_text,score
2,2,Northern,Delhi,Delhi,Akshardham Temple,Temple,2005,5.0,4.6,60,Yes,NaN,Religious,No,0.40,Afternoon,Delhi Akshardham Temple Temple,0.161583
6,6,Northern,Delhi,Delhi,Lotus Temple,Temple,1986,1.0,4.5,0,Yes,Monday,Religious,Yes,0.59,Evening,Delhi Lotus Temple Temple,0.156781
7,7,Northern,Delhi,Delhi,Red Fort,Fort,1648,2.0,4.5,35,Yes,NaN,Historical,Yes,1.50,Afternoon,Delhi Red Fort Fort,0.144579
10,10,Northern,Delhi,Delhi,Garden of Five Senses,Park,2003,2.0,4.1,35,Yes,NaN,Botanical,Yes,0.23,Morning,Delhi Garden of Five Senses Park,0.141556
11,11,Northern,Delhi,Delhi,Lodhi Garden,Park,1500,1.0,4.5,0,Yes,NaN,Botanical,Yes,0.48,All,Delhi Lodhi Garden Park,0.141556
13,13,Northern,Delhi,Delhi,National Zoological Park,Zoo,1959,3.0,4.1,80,Yes,Friday,Environmental,Yes,0.41,All,Delhi National Zoological Park Zoo,0.140219
9,9,Northern,Delhi,Delhi,Sunder Nursery,Park,1600,2.0,4.6,0,Yes,NaN,Botanical,Yes,0.16,Afternoon,Delhi Sunder Nursery Park,0.138039
14,14,Northern,Delhi,Delhi,Qutub Minar,Monument,1192,1.0,4.5,35,Yes,NaN,Historical,Yes,1.37,Afternoon,Delhi Qutub Minar Monument,0.134975
5,5,Northern,Delhi,Delhi,Chandni Chowk,Market,1700,3.0,4.2,0,Yes,Sunday,Market,Yes,0.25,Afternoon,Delhi Chandni Chowk Market,0.134157
4,4,Northern,Delhi,Delhi,Jantar Mantar,Observatory,1724,2.0,4.2,15,Yes,NaN,Scientific,Yes,0.31,Morning,Delhi Jantar Mantar Observatory,0.130040


In [14]:


stations = pd.read_csv("stations.csv")

print(stations.shape)
stations.head()

(553, 11)


,id,station_code,city,state_code,pin_code,latitude,longitude,elevation_m,topo_complexity,coastal_proximity,valley_factor
0,1,DL01,Dwarka,DL,110078,28.610948,77.038456,215,0.7,0.000000,0.80
1,2,DL02,New Delhi,DL,110006,28.628191,77.241044,212,0.7,0.000000,0.80
2,3,DL03,Delhi,DL,110042,28.750815,77.116277,217,0.7,0.000000,0.90
3,4,CG01,Pawaras,CG,486865,18.388855,81.683361,212,0.5,0.370772,0.60
4,5,RJ01,Alwar,RJ,356172,27.550006,76.611703,286,0.3,0.000000,0.95


In [15]:
print(stations.columns.tolist())

['id', 'station_code', 'city', 'state_code', 'pin_code', 'latitude', 'longitude', 'elevation_m', 'topo_complexity', 'coastal_proximity', 'valley_factor']


In [16]:
print(type(stations))
print(stations.shape)
print(stations.columns.tolist())

<class 'pandas.DataFrame'>
(553, 11)
['id', 'station_code', 'city', 'state_code', 'pin_code', 'latitude', 'longitude', 'elevation_m', 'topo_complexity', 'coastal_proximity', 'valley_factor']


In [17]:
stations.info()

<class 'pandas.DataFrame'>
RangeIndex: 553 entries, 0 to 552
Data columns (total 11 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 553 non-null    int64  
 1   station_code       553 non-null    str    
 2   city               553 non-null    str    
 3   state_code         553 non-null    str    
 4   pin_code           553 non-null    int64  
 5   latitude           553 non-null    float64
 6   longitude          552 non-null    float64
 7   elevation_m        553 non-null    int64  
 8   topo_complexity    553 non-null    float64
 9   coastal_proximity  553 non-null    float64
 10  valley_factor      553 non-null    float64
dtypes: float64(5), int64(3), str(3)
memory usage: 54.9 KB


In [18]:
stations.isnull().sum().sort_values(ascending=False)

longitude            1
station_code         0
id                   0
city                 0
state_code           0
pin_code             0
latitude             0
elevation_m          0
topo_complexity      0
coastal_proximity    0
valley_factor        0
dtype: int64

In [19]:
stations.describe(include="all")

,id,station_code,city,state_code,pin_code,latitude,longitude,elevation_m,topo_complexity,coastal_proximity,valley_factor
count,553.000000,553,553,553,553.000000,553.000000,552.000000,553.000000,553.000000,553.000000,553.000000
unique,NaN,547,263,20,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,UN01,Delhi,UN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,2,36,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,277.000000,NaN,NaN,NaN,426284.602170,21.698306,78.433245,290.229656,0.441953,0.296525,0.607595
std,159.781622,NaN,NaN,NaN,190967.686753,5.579952,4.407306,264.996486,0.156858,0.310169,0.227256
min,1.000000,NaN,NaN,NaN,13167.000000,8.178860,69.603000,1.000000,0.200000,0.000000,0.050000
25%,139.000000,NaN,NaN,NaN,297495.000000,18.621009,75.820736,55.000000,0.300000,0.000000,0.400000
50%,277.000000,NaN,NaN,NaN,451758.000000,22.687581,77.686029,221.000000,0.400000,0.200250,0.600000
75%,415.000000,NaN,NaN,NaN,521665.000000,26.210445,79.370366,484.000000,0.600000,0.583857,0.800000


In [20]:
for i, col in enumerate(stations.columns):
    print(i, ":", col)

0 : id
1 : station_code
2 : city
3 : state_code
4 : pin_code
5 : latitude
6 : longitude
7 : elevation_m
8 : topo_complexity
9 : coastal_proximity
10 : valley_factor


In [21]:
stations.head(10).T

,0,1,2,3,4,5,6,7,8,9
id,1,2,3,4,5,6,7,8,9,10
station_code,DL01,DL02,DL03,CG01,RJ01,RJ02,UN01,GJ01,WB01,RJ03
city,Dwarka,New Delhi,Delhi,Pawaras,Alwar,Bhiwadi,RSPCB,Shrinathgadh,Indira Colony,Jaipur
state_code,DL,DL,DL,CG,RJ,RJ,UN,GJ,WB,RJ
pin_code,110078,110006,110042,486865,356172,108164,483765,369067,705353,397944
latitude,28.610948,28.628191,28.750815,18.388855,27.550006,28.202045,22.687581,21.862938,22.703568,26.894094
longitude,77.038456,77.241044,77.116277,81.683361,76.611703,76.817954,79.370366,70.909236,88.536384,75.79277
elevation_m,215,212,217,212,286,252,484,131,7,417
topo_complexity,0.7,0.7,0.7,0.5,0.3,0.6,0.4,0.3,0.4,0.3
coastal_proximity,0.0,0.0,0.0,0.370772,0.0,0.0,0.0,0.556947,0.625303,0.0


In [22]:
stations.head()

,id,station_code,city,state_code,pin_code,latitude,longitude,elevation_m,topo_complexity,coastal_proximity,valley_factor
0,1,DL01,Dwarka,DL,110078,28.610948,77.038456,215,0.7,0.000000,0.80
1,2,DL02,New Delhi,DL,110006,28.628191,77.241044,212,0.7,0.000000,0.80
2,3,DL03,Delhi,DL,110042,28.750815,77.116277,217,0.7,0.000000,0.90
3,4,CG01,Pawaras,CG,486865,18.388855,81.683361,212,0.5,0.370772,0.60
4,5,RJ01,Alwar,RJ,356172,27.550006,76.611703,286,0.3,0.000000,0.95


In [23]:
stations = stations.dropna(
    subset=["latitude", "longitude"]
).copy()

print(stations.shape)

(552, 11)


In [24]:
from sklearn.neighbors import NearestNeighbors
import numpy as np

stations_coords = stations[
    ["latitude", "longitude"]
].values

stations_coords_rad = np.radians(stations_coords)

nearest_stations_model = NearestNeighbors(
    n_neighbors=1,
    metric="haversine"
)

nearest_stations_model.fit(stations_coords_rad)

print("Nearest-station model ready.")

Nearest-station model ready.


In [25]:
ranchi_lat = 23.3441
ranchi_lon = 85.3096

point = np.radians(
    [[ranchi_lat, ranchi_lon]]
)

distance, index = nearest_stations_model.kneighbors(point)

nearest = stations.iloc[index[0][0]]

print(nearest)

id                           26
station_code               JH01
city                 Jamshedpur
state_code                   JH
pin_code                 701915
latitude              22.806013
longitude             86.193541
elevation_m                 154
topo_complexity             0.3
coastal_proximity      0.429949
valley_factor               0.6
Name: 25, dtype: object


In [26]:
aqi_folder = "../data/aqi_data"

files = glob.glob(
    os.path.join(aqi_folder, "**", "*.csv"),
    recursive=True
)

print("Number of CSV files:", len(files))

files[:10]

Number of CSV files: 0


[]

In [27]:
%whos

Variable                 Type                Data/Info
------------------------------------------------------
NearestNeighbors         ABCMeta             <class 'sklearn.neighbors<...>rvised.NearestNeighbors'>
RandomForestRegressor    ABCMeta             <class 'sklearn.ensemble.<...>t.RandomForestRegressor'>
TfidfVectorizer          type                <class 'sklearn.feature_e<...>on.text.TfidfVectorizer'>
aqi_folder               str                 ../data/aqi_data
col                      str                 valley_factor
cosine_similarity        function            <function cosine_similari<...>ty at 0x0000015AF5BA85E0>
distance                 ndarray             1x1: 1 elems, type `float64`, 8 bytes
files                    list                n=0
glob                     module              <module 'glob' from 'C:\\<...>Python313\\Lib\\glob.py'>
i                        int                 10
index                    ndarray             1x1: 1 elems, type `int64`, 8 bytes
job

In [28]:
import os

downloads = r"C:\Users\Aryan kumar\Downloads"

for name in os.listdir(downloads):
    if "archive" in name.lower():
        path = os.path.join(downloads, name)
        print(name, "->", "FOLDER" if os.path.isdir(path) else "FILE")

archive -> FOLDER
archive (1) -> FOLDER
archive (1).zip -> FILE
archive (2) -> FOLDER
archive (2).zip -> FILE
archive (3) -> FOLDER
archive (3).zip -> FILE
archive (4) -> FOLDER
archive (4).zip -> FILE
archive.zip -> FILE


In [29]:
import zipfile
import os

downloads = r"C:\Users\Aryan kumar\Downloads"

aqi_zip = os.path.join(
    downloads,
    "archive (4).zip"
)

aqi_extract = os.path.join(
    downloads,
    "AQI_Data"
)

os.makedirs(aqi_extract, exist_ok=True)

with zipfile.ZipFile(aqi_zip, "r") as zip_ref:
    zip_ref.extractall(aqi_extract)

print("AQI dataset extracted successfully!")
print(aqi_extract)

AQI dataset extracted successfully!
C:\Users\Aryan kumar\Downloads\AQI_Data


In [30]:
import glob
import os

aqi_files = glob.glob(
    os.path.join(
        aqi_extract,
        "**",
        "*.csv"
    ),
    recursive=True
)

print("Total AQI CSV files:", len(aqi_files))

for file in aqi_files[:10]:
    print(file)

Total AQI CSV files: 547
C:\Users\Aryan kumar\Downloads\AQI_Data\stations.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP01.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP02.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP03.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP04.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP05.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP06.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP07.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP08.csv
C:\Users\Aryan kumar\Downloads\AQI_Data\Andhra_Pradesh\AP09.csv


In [31]:
sample_file = aqi_files[0]

sample_aqi = pd.read_csv(sample_file)

print("File:", sample_file)
print("Shape:", sample_aqi.shape)

sample_aqi.head()

File: C:\Users\Aryan kumar\Downloads\AQI_Data\stations.csv
Shape: (553, 11)


,id,station_code,city,state_code,pin_code,latitude,longitude,elevation_m,topo_complexity,coastal_proximity,valley_factor
0,1,DL01,Dwarka,DL,110078,28.610948,77.038456,215,0.7,0.000000,0.80
1,2,DL02,New Delhi,DL,110006,28.628191,77.241044,212,0.7,0.000000,0.80
2,3,DL03,Delhi,DL,110042,28.750815,77.116277,217,0.7,0.000000,0.90
3,4,CG01,Pawaras,CG,486865,18.388855,81.683361,212,0.5,0.370772,0.60
4,5,RJ01,Alwar,RJ,356172,27.550006,76.611703,286,0.3,0.000000,0.95


In [32]:
print(sample_aqi.columns.tolist())

['id', 'station_code', 'city', 'state_code', 'pin_code', 'latitude', 'longitude', 'elevation_m', 'topo_complexity', 'coastal_proximity', 'valley_factor']


In [33]:
sample_aqi.head().T

,0,1,2,3,4
id,1,2,3,4,5
station_code,DL01,DL02,DL03,CG01,RJ01
city,Dwarka,New Delhi,Delhi,Pawaras,Alwar
state_code,DL,DL,DL,CG,RJ
pin_code,110078,110006,110042,486865,356172
latitude,28.610948,28.628191,28.750815,18.388855,27.550006
longitude,77.038456,77.241044,77.116277,81.683361,76.611703
elevation_m,215,212,217,212,286
topo_complexity,0.7,0.7,0.7,0.5,0.3
coastal_proximity,0.0,0.0,0.0,0.370772,0.0


In [34]:
for file in aqi_files[:10]:
    df = pd.read_csv(file, nrows=2)
    print(
        os.path.basename(file),
        "->",
        df.columns.tolist()
    )

stations.csv -> ['id', 'station_code', 'city', 'state_code', 'pin_code', 'latitude', 'longitude', 'elevation_m', 'topo_complexity', 'coastal_proximity', 'valley_factor']
AP01.csv -> ['pm2.5', 'pm10', 'no', 'no2', 'so2', 'co', 'nox', 'ozone', 'at', 'rh', 'ws', 'vws', 'wd', 'bp', 'timestamp', 'nh3', 'benzene', 'toluene', 'xylene', 'mp_xylene']
AP02.csv -> ['pm2.5', 'pm10', 'no', 'no2', 'so2', 'co', 'nox', 'ozone', 'at', 'ws', 'wd', 'bp', 'timestamp', 'nh3', 'benzene', 'toluene']
AP03.csv -> ['pm2.5', 'pm10', 'no', 'no2', 'so2', 'co', 'nox', 'ozone', 'at', 'rh', 'ws', 'vws', 'wd', 'bp', 'timestamp', 'nh3', 'benzene', 'eth_benzene', 'toluene', 'mp_xylene', 'o_xylene']
AP04.csv -> ['pm2.5', 'pm10', 'no', 'no2', 'so2', 'co', 'nox', 'ozone', 'at', 'rh', 'ws', 'vws', 'wd', 'timestamp', 'nh3', 'benzene', 'toluene', 'xylene']
AP05.csv -> ['pm2.5', 'pm10', 'no', 'no2', 'so2', 'co', 'nox', 'ozone', 'at', 'rh', 'ws', 'vws', 'wd', 'bp', 'timestamp', 'nh3', 'benzene', 'toluene', 'xylene']
AP06.csv ->

In [35]:
selected_files = aqi_files[:50]

print("Using", len(selected_files), "files for initial model")

Using 50 files for initial model


In [36]:
aqi_list = []

for file in selected_files:
    try:
        df = pd.read_csv(file)
        aqi_list.append(df)
        print("Loaded:", os.path.basename(file), df.shape)
    except Exception as e:
        print("Error:", file, e)

Loaded: stations.csv (553, 11)
Loaded: AP01.csv (214538, 20)
Loaded: AP02.csv (103464, 16)
Loaded: AP03.csv (66574, 21)
Loaded: AP04.csv (14998, 18)
Loaded: AP05.csv (239122, 19)
Loaded: AP06.csv (152262, 20)
Loaded: AP07.csv (157009, 20)
Loaded: AP08.csv (104559, 18)
Loaded: AP09.csv (65260, 20)
Loaded: AP10.csv (70722, 18)
Loaded: AP11.csv (53680, 19)
Loaded: AP12.csv (62270, 20)
Loaded: AP13.csv (44844, 20)
Loaded: AP14.csv (26809, 20)
Loaded: AS01.csv (76897, 17)
Loaded: AS02.csv (17964, 9)
Loaded: AS03.csv (199152, 18)
Loaded: AS04.csv (77250, 18)
Loaded: AS05.csv (69654, 18)
Loaded: AS06.csv (65079, 18)
Loaded: AS07.csv (63787, 18)
Loaded: BR01.csv (231947, 9)
Loaded: BR02.csv (195400, 17)
Loaded: BR03.csv (271410, 17)
Loaded: BR04.csv (268008, 17)
Loaded: BR05.csv (168976, 20)
Loaded: BR06.csv (173796, 20)
Loaded: BR07.csv (169970, 20)
Loaded: BR08.csv (148284, 18)
Loaded: BR09.csv (163284, 20)
Loaded: BR10.csv (162395, 20)
Loaded: BR11.csv (148542, 20)
Loaded: BR12.csv (117174,

In [37]:
aqi_data = pd.concat(
    aqi_list,
    ignore_index=True
)

print("Final shape:", aqi_data.shape)

Final shape: (5637586, 33)


In [38]:
aqi_data.head()
aqi_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5637586 entries, 0 to 5637585
Data columns (total 33 columns):
 #   Column             Dtype  
---  ------             -----  
 0   id                 float64
 1   station_code       str    
 2   city               str    
 3   state_code         str    
 4   pin_code           float64
 5   latitude           float64
 6   longitude          float64
 7   elevation_m        float64
 8   topo_complexity    float64
 9   coastal_proximity  float64
 10  valley_factor      float64
 11  pm2.5              float64
 12  pm10               float64
 13  no                 float64
 14  no2                float64
 15  so2                float64
 16  co                 float64
 17  nox                float64
 18  ozone              float64
 19  at                 float64
 20  rh                 float64
 21  ws                 float64
 22  vws                float64
 23  wd                 float64
 24  bp                 float64
 25  timestamp          str    
 2

In [39]:
sample_aqi.head().T

,0,1,2,3,4
id,1,2,3,4,5
station_code,DL01,DL02,DL03,CG01,RJ01
city,Dwarka,New Delhi,Delhi,Pawaras,Alwar
state_code,DL,DL,DL,CG,RJ
pin_code,110078,110006,110042,486865,356172
latitude,28.610948,28.628191,28.750815,18.388855,27.550006
longitude,77.038456,77.241044,77.116277,81.683361,76.611703
elevation_m,215,212,217,212,286
topo_complexity,0.7,0.7,0.7,0.5,0.3
coastal_proximity,0.0,0.0,0.0,0.370772,0.0


In [ ]:
aqi_list = []

for file in aqi_files:
    try:
        df = pd.read_csv(file)
        
        
        df["source_file"] = os.path.basename(file)
        
        aqi_list.append(df)
        
    except Exception as e:
        print("Error:", file, e)

aqi_data = pd.concat(aqi_list, ignore_index=True)

print("Final AQI dataset shape:", aqi_data.shape)
aqi_data.head()

In [41]:
print(aqi_data.info())

<class 'pandas.DataFrame'>
RangeIndex: 5637586 entries, 0 to 5637585
Data columns (total 33 columns):
 #   Column             Dtype  
---  ------             -----  
 0   id                 float64
 1   station_code       str    
 2   city               str    
 3   state_code         str    
 4   pin_code           float64
 5   latitude           float64
 6   longitude          float64
 7   elevation_m        float64
 8   topo_complexity    float64
 9   coastal_proximity  float64
 10  valley_factor      float64
 11  pm2.5              float64
 12  pm10               float64
 13  no                 float64
 14  no2                float64
 15  so2                float64
 16  co                 float64
 17  nox                float64
 18  ozone              float64
 19  at                 float64
 20  rh                 float64
 21  ws                 float64
 22  vws                float64
 23  wd                 float64
 24  bp                 float64
 25  timestamp          str    
 2

In [42]:
print(aqi_data.isnull().sum())

id                   5637033
station_code         5637033
city                 5637033
state_code           5637033
pin_code             5637033
latitude             5637033
longitude            5637034
elevation_m          5637033
topo_complexity      5637033
coastal_proximity    5637033
valley_factor        5637033
pm2.5                 486438
pm10                  946132
no                    535058
no2                   496566
so2                   468616
co                    336941
nox                   241328
ozone                 589412
at                   1057083
rh                    747284
ws                    584883
vws                  4583083
wd                    607561
bp                   2284212
timestamp                553
nh3                  1428198
benzene               771255
toluene              1525667
xylene               2912976
mp_xylene            2537246
eth_benzene          2721809
o_xylene             5577268
dtype: int64


In [43]:
aqi_data.describe(include="all").T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
id,553.0,NaN,NaN,NaN,277.0,159.781622,1.0,139.0,277.0,415.0,553.0
station_code,553,547,UN01,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
city,553,263,Delhi,36,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state_code,553,20,UN,106,NaN,NaN,NaN,NaN,NaN,NaN,NaN
pin_code,553.0,NaN,NaN,NaN,426284.60217,190967.686753,13167.0,297495.0,451758.0,521665.0,811320.0
latitude,553.0,NaN,NaN,NaN,21.698306,5.579952,8.17886,18.621009,22.687581,26.210445,34.06944
longitude,552.0,NaN,NaN,NaN,78.433245,4.407306,69.603,75.820736,77.686029,79.370366,95.165024
elevation_m,553.0,NaN,NaN,NaN,290.229656,264.996486,1.0,55.0,221.0,484.0,2064.0
topo_complexity,553.0,NaN,NaN,NaN,0.441953,0.156858,0.2,0.3,0.4,0.6,1.0
coastal_proximity,553.0,NaN,NaN,NaN,0.296525,0.310169,0.0,0.0,0.20025,0.583857,0.991539


In [44]:
possible_pollutants = [
    "PM2.5", "PM10", "NO2", "SO2",
    "CO", "O3", "NH3", "Pb"
]

pollutant_columns = [
    col for col in aqi_data.columns
    if any(p.lower() == str(col).lower() for p in possible_pollutants)
]

print("Detected pollutant columns:")
print(pollutant_columns)

Detected pollutant columns:
['pm2.5', 'pm10', 'no2', 'so2', 'co', 'nh3']


In [45]:
for col in pollutant_columns:
    aqi_data[col] = pd.to_numeric(
        aqi_data[col],
        errors="coerce"
    )

print(aqi_data[pollutant_columns].dtypes)

pm2.5    float64
pm10     float64
no2      float64
so2      float64
co       float64
nh3      float64
dtype: object


In [46]:
aqi_data = aqi_data.dropna(
    subset=pollutant_columns,
    how="all"
).copy()

print("Shape after cleaning:", aqi_data.shape)

MemoryError: Unable to allocate 42.9 MiB for an array with shape (5622231, 1) and data type float64

In [47]:
aqi_columns = [
    col for col in aqi_data.columns
    if "aqi" in str(col).lower()
]

print("Possible AQI columns:")
print(aqi_columns)

Possible AQI columns:
[]
